In [1]:
# Import Libraries

import pandas as pd
import numpy as np
from pathlib import Path



In [2]:
# Define the dataset path
file_path = Path("Advertising.csv")

# Check whether the file exists
if not file_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {file_path.resolve()}\n"
        "Place Advertising.csv in your working directory "
        "or update file_path to its full location."
    )

# Load the CSV file
df = pd.read_csv(file_path)

#  Display the first five rows
print("Dataset loaded successfully!")
display(df.head())

Dataset loaded successfully!


,Unnamed: 0,TV,Radio,Newspaper,Sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [3]:
# Inspect dimensions, column names, and data types
# Dataset dimensions
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Column names
print("\nColumn names:")
print(df.columns.tolist())

# Data types, non-null counts, and memory usage
print("\nDataset information:")
df.info()

# Last five rows
display(df.tail())

Number of rows: 200
Number of columns: 5

Column names:
['Unnamed: 0', 'TV', 'Radio', 'Newspaper', 'Sales']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  200 non-null    int64  
 1   TV          200 non-null    float64
 2   Radio       200 non-null    float64
 3   Newspaper   200 non-null    float64
 4   Sales       200 non-null    float64
dtypes: float64(4), int64(1)
memory usage: 7.9 KB


,Unnamed: 0,TV,Radio,Newspaper,Sales
195,196,38.2,3.7,13.8,7.6
196,197,94.2,4.9,8.1,9.7
197,198,177.0,9.3,6.4,12.8
198,199,283.6,42.0,66.2,25.5
199,200,232.1,8.6,8.7,13.4


In [4]:
# summary statistics
# Descriptive statistics for all numeric columns
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,200.0,100.5000,57.879185,1.0,50.750,100.50,150.250,200.0
TV,200.0,147.0425,85.854236,0.7,74.375,149.75,218.825,296.4
Radio,200.0,23.2640,14.846809,0.0,9.975,22.90,36.525,49.6
Newspaper,200.0,30.5540,21.778621,0.3,12.750,25.75,45.100,114.0
Sales,200.0,14.0225,5.217457,1.6,10.375,12.90,17.400,27.0


In [5]:
# missing values and duplicate records
# Count missing values in each column
print("Missing values per column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Total missing values:", df.isnull().sum().sum())

# Count exact duplicate rows
print("Duplicate rows:", df.duplicated().sum())

Missing values per column:


,Missing Values
Unnamed: 0,0
TV,0
Radio,0
Newspaper,0
Sales,0


Total missing values: 0
Duplicate rows: 0


In [6]:
# Identify the unnecessary index column
# Check whether the column is sequential
index_column = "Unnamed: 0"

is_sequential = df[index_column].equals(
    pd.Series(range(1, len(df) + 1))
)

print("Index column exists:", index_column in df.columns)
print("Values are unique:", df[index_column].is_unique)
print("Values are sequential from 1 to 200:", is_sequential)

Index column exists: True
Values are unique: True
Values are sequential from 1 to 200: True


In [7]:
# Audit report
audit_report = pd.DataFrame({
    "Check": [
        "Number of rows",
        "Number of columns",
        "Total missing values",
        "Duplicate rows",
        "Numeric columns",
        "Sequential index column",
        "Negative values in substantive variables"
    ],
    "Result": [
        df.shape[0],
        df.shape[1],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        int(df.select_dtypes(include=np.number).shape[1]),
        is_sequential,
        int((df[["TV", "Radio", "Newspaper", "Sales"]] < 0).sum().sum())
    ]
})

display(audit_report)

,Check,Result
0,Number of rows,200
1,Number of columns,5
2,Total missing values,0
3,Duplicate rows,0
4,Numeric columns,5
5,Sequential index column,True
6,Negative values in substantive variables,0


In [8]:
# Clean and Transform the Data
# Define the required columns
required_columns = ["TV", "Radio", "Newspaper", "Sales"]

# Clean whitespace from column names
df.columns = df.columns.str.strip()

# Validate the required columns
missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("Original dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Original dataset shape: (200, 5)
Columns: ['Unnamed: 0', 'TV', 'Radio', 'Newspaper', 'Sales']


In [9]:
# Remove the unnecessary index column
# Remove the saved index column if it exists
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Confirm the remaining columns
print("Columns after removing index:")
print(df.columns.tolist())

print("\nDataset shape:", df.shape)
display(df.head())

Columns after removing index:
['TV', 'Radio', 'Newspaper', 'Sales']

Dataset shape: (200, 4)


,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [10]:
# Validate data types and missing values
# Convert the required columns to numeric values.
# Invalid text values will raise an error rather than be silently discarded.
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors="raise")

# Check missing values
print("Missing values:")
print(df[required_columns].isnull().sum())

print("\nTotal missing values:", df[required_columns].isnull().sum().sum())

# Check data types
print("\nData types:")
print(df[required_columns].dtypes)

Missing values:
TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64

Total missing values: 0

Data types:
TV           float64
Radio        float64
Newspaper    float64
Sales        float64
dtype: object


In [11]:
# Check and handle duplicate records
# Count exact duplicate records
duplicate_count = df.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

# Remove exact duplicates only if they exist
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicate rows removed:", duplicate_count)
else:
    print("No duplicate rows to remove.")

Duplicate rows found: 0
No duplicate rows to remove.


In [12]:
# Validate values and investigate potential outliers
# Check negative values
negative_counts = (df[required_columns] < 0).sum()

print("Negative values by column:")
print(negative_counts)

# Calculate potential outliers using the IQR method
outlier_results = []

for column in required_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (df[column] < lower_bound) | (df[column] > upper_bound)

    outlier_results.append({
        "Column": column,
        "Lower bound": lower_bound,
        "Upper bound": upper_bound,
        "Potential outliers": int(mask.sum())
    })

outlier_report = pd.DataFrame(outlier_results)
display(outlier_report)

Negative values by column:
TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64


,Column,Lower bound,Upper bound,Potential outliers
0,TV,-142.3000,435.5000,0
1,Radio,-29.8500,76.3500,0
2,Newspaper,-35.7750,93.6250,2
3,Sales,-0.1625,27.9375,0


In [13]:
# Save the cleaned dataset
# Confirm the final feature and target columns
feature_columns = ["TV", "Radio", "Newspaper"]
target_column = "Sales"

cleaned_df = df[feature_columns + [target_column]].copy()

# Final validation before saving
assert cleaned_df.columns.tolist() == required_columns
assert cleaned_df.isnull().sum().sum() == 0
assert cleaned_df.duplicated().sum() == 0

# Save the cleaned dataset
output_path = Path("Advertising_Cleaned.csv")
cleaned_df.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path.resolve())
print("Final shape:", cleaned_df.shape)

display(cleaned_df.head())

Cleaned dataset saved to: /content/Advertising_Cleaned.csv
Final shape: (200, 4)


,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [14]:
# Verify the saved file
# Reload the cleaned dataset
verified_df = pd.read_csv(output_path)

# Verify structure and contents
assert verified_df.shape == cleaned_df.shape
assert verified_df.columns.tolist() == required_columns
assert verified_df.equals(cleaned_df)

print("Verification successful!")
print("Rows:", verified_df.shape[0])
print("Columns:", verified_df.shape[1])
print("Missing values:", verified_df.isnull().sum().sum())
print("Duplicate rows:", verified_df.duplicated().sum())
print("Saved file size:", output_path.stat().st_size, "bytes")

Verification successful!
Rows: 200
Columns: 4
Missing values: 0
Duplicate rows: 0
Saved file size: 4012 bytes
